In [ ]:
import pandas as pd
import requests
import yfinance as yf
from datetime import datetime, timedelta
import numpy as np

class MultiSourceSentimentCollector:
    """
    Collect pre-calculated sentiment data from multiple sources for HARLF portfolio
    """
    
    def __init__(self, assets, start_date="2015-01-01", end_date="2024-12-31"):
        self.assets = assets
        self.start_date = start_date
        self.end_date = end_date
        self.sentiment_data = {}
        
    def collect_alpha_vantage_sentiment(self, api_key, symbol):
        """
        Collect sentiment from Alpha Vantage News Sentiment API
        """
        base_url = "https://www.alphavantage.co/query"
        
        # Get news sentiment
        params = {
            'function': 'NEWS_SENTIMENT',
            'tickers': symbol,
            'time_from': self.start_date.replace('-', '') + 'T0000',
            'time_to': self.end_date.replace('-', '') + 'T2359',
            'limit': 1000,
            'apikey': api_key
        }
        
        try:
            response = requests.get(base_url, params=params)
            data = response.json()
            
            if 'feed' in data:
                sentiment_scores = []
                dates = []
                
                for article in data['feed']:
                    if 'ticker_sentiment' in article:
                        for ticker_data in article['ticker_sentiment']:
                            if ticker_data['ticker'] == symbol:
                                sentiment_scores.append(float(ticker_data['ticker_sentiment_score']))
                                dates.append(pd.to_datetime(article['time_published']))
                
                # Convert to monthly data
                df = pd.DataFrame({
                    'date': dates,
                    'sentiment': sentiment_scores
                }).set_index('date')
                
                monthly_sentiment = df.resample('M').mean()
                return monthly_sentiment['sentiment']
                
        except Exception as e:
            print(f"Error collecting Alpha Vantage sentiment for {symbol}: {e}")
            return None
    
    def collect_yahoo_finance_sentiment(self, symbol):
        """
        Extract sentiment indicators from Yahoo Finance news
        Note: This is a simplified approach - Yahoo doesn't provide direct sentiment scores
        """
        try:
            ticker = yf.Ticker(symbol)
            news = ticker.news
            
            # This would need additional NLP processing
            # For demonstration - you'd integrate with FinBERT here
            print(f"Found {len(news)} articles for {symbol}")
            return None  # Placeholder
            
        except Exception as e:
            print(f"Error collecting Yahoo sentiment for {symbol}: {e}")
            return None
    
    def collect_quandl_sentiment(self, api_key, symbol):
        """
        Collect from Quandl alternative data sources
        """
        # Example endpoint (actual endpoints vary by data provider)
        url = f"https://www.quandl.com/api/v3/datasets/NS1/{symbol}_SENTIMENT.json"
        
        params = {
            'api_key': api_key,
            'start_date': self.start_date,
            'end_date': self.end_date
        }
        
        try:
            response = requests.get(url, params=params)
            data = response.json()
            
            if 'dataset' in data:
                df = pd.DataFrame(
                    data['dataset']['data'],
                    columns=data['dataset']['column_names']
                )
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)
                
                # Resample to monthly
                monthly_sentiment = df['Sentiment'].resample('M').mean()
                return monthly_sentiment
                
        except Exception as e:
            print(f"Error collecting Quandl sentiment for {symbol}: {e}")
            return None
    
    def validate_sentiment_quality(self, sentiment_series, symbol):
        """
        Validate sentiment data quality based on HARLF criteria
        """
        if sentiment_series is None or len(sentiment_series) < 24:
            return False, "Insufficient data"
        
        # Check for reasonable sentiment range
        if sentiment_series.min() < -1.5 or sentiment_series.max() > 1.5:
            return False, "Sentiment values out of expected range"
        
        # Check for excessive missing values
        missing_pct = sentiment_series.isna().sum() / len(sentiment_series)
        if missing_pct > 0.3:
            return False, f"Too many missing values: {missing_pct:.1%}"
        
        # Check volatility (avoid noisy sentiment)
        sentiment_volatility = sentiment_series.rolling(6).std().mean()
        if sentiment_volatility > 0.4:
            return False, f"Sentiment too volatile: {sentiment_volatility:.3f}"
        
        return True, "Quality check passed"
    
    def collect_all_sources(self, alpha_vantage_key=None, quandl_key=None):
        """
        Collect sentiment from all available sources for each asset
        """
        results = {}
        
        for symbol in self.assets:
            print(f"\n=== Collecting sentiment for {symbol} ===")
            symbol_results = {}
            
            # Try Alpha Vantage first (highest quality)
            if alpha_vantage_key:
                av_sentiment = self.collect_alpha_vantage_sentiment(alpha_vantage_key, symbol)
                if av_sentiment is not None:
                    is_valid, msg = self.validate_sentiment_quality(av_sentiment, symbol)
                    symbol_results['alpha_vantage'] = {
                        'data': av_sentiment,
                        'valid': is_valid,
                        'message': msg
                    }
                    print(f"  ✅ Alpha Vantage: {msg}")
                else:
                    print(f"  ❌ Alpha Vantage: No data available")
            
            # Try Quandl as backup
            if quandl_key:
                quandl_sentiment = self.collect_quandl_sentiment(quandl_key, symbol)
                if quandl_sentiment is not None:
                    is_valid, msg = self.validate_sentiment_quality(quandl_sentiment, symbol)
                    symbol_results['quandl'] = {
                        'data': quandl_sentiment,
                        'valid': is_valid,
                        'message': msg
                    }
                    print(f"  ✅ Quandl: {msg}")
                else:
                    print(f"  ❌ Quandl: No data available")
            
            # Yahoo Finance as last resort (requires additional processing)
            yahoo_sentiment = self.collect_yahoo_finance_sentiment(symbol)
            if yahoo_sentiment is not None:
                symbol_results['yahoo'] = yahoo_sentiment
                print(f"  📰 Yahoo: Raw news data collected (needs NLP processing)")
            
            results[symbol] = symbol_results
        
        return results
    
    def create_final_sentiment_dataset(self, collected_data):
        """
        Create final sentiment dataset prioritizing best available sources
        """
        final_sentiment = pd.DataFrame(index=pd.date_range(
            start=self.start_date, end=self.end_date, freq='M'
        ))
        
        source_priority = ['alpha_vantage', 'quandl', 'yahoo']
        sentiment_quality_report = {}
        
        for symbol in self.assets:
            best_sentiment = None
            best_source = None
            
            # Try sources in priority order
            for source in source_priority:
                if (symbol in collected_data and 
                    source in collected_data[symbol] and
                    collected_data[symbol][source].get('valid', False)):
                    
                    best_sentiment = collected_data[symbol][source]['data']
                    best_source = source
                    break
            
            if best_sentiment is not None:
                # Align with monthly index
                final_sentiment[symbol] = best_sentiment
                sentiment_quality_report[symbol] = {
                    'source': best_source,
                    'coverage': best_sentiment.notna().sum(),
                    'quality': 'good'
                }
                print(f"✅ {symbol}: Using {best_source} sentiment")
            else:
                # Fill with zeros for assets without useful sentiment
                final_sentiment[symbol] = 0.0
                sentiment_quality_report[symbol] = {
                    'source': 'none',
                    'coverage': 0,
                    'quality': 'filtered_out'
                }
                print(f"⭕ {symbol}: No useful sentiment found - using zero fill")
        
        return final_sentiment, sentiment_quality_report

# Usage example
PERSONAL_ASSETS = [
    'RDDT', 'NVDA', 'SMR', 'MU', 'MRVL', 'MSFT', 'ASML', 'AEM',
    'AMD', 'VERU', 'AI', 'GOOGL', 'INGM', 'PLUG', 'IONQ', 'CHYM', 'RGTI', 'ARBE'
]

# Initialize collector
collector = MultiSourceSentimentCollector(
    assets=PERSONAL_ASSETS,
    start_date="2015-01-01",
    end_date="2024-12-31"
)

# Collect from all sources (add your API keys)
sentiment_results = collector.collect_all_sources(
    alpha_vantage_key="e97eb96cba2d4f0e9acd506f079e0944",
    quandl_key="YOUR_QUANDL_KEY"
)

# Create final dataset
final_sentiment_data, quality_report = collector.create_final_sentiment_dataset(sentiment_results)

print("\n=== SENTIMENT COLLECTION SUMMARY ===")
for asset, report in quality_report.items():
    print(f"{asset}: {report['source']} - {report['coverage']} months - {report['quality']}")

# Save results
final_sentiment_data.to_csv('sentiment_data_precalculated.csv')
print(f"\n✅ Sentiment data saved with shape: {final_sentiment_data.shape}")